# Smart MCQ Solver — Milestone 4
**Email:** 23f3001763@ds.study.iitm.ac.in



In [1]:
# Run this cell first to ensure all required libraries are installed on Kaggle
!pip install -q datasets transformers peft accelerate torch scikit-learn pandas

In [9]:
!pip install -q datasets transformers peft accelerate torch scikit-learn pandas "torchao>=0.16.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 29.4 MB/s eta 0:00:0000:0100:01


In [3]:
import os
import torch
import random
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer, DefaultDataCollator, set_seed
from peft import get_peft_model, LoraConfig, TaskType

# ── Set Global Seeds for Reproducibility ──────────────────────────────────────
# This ensures everyone gets the exact same probability for Q10
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    set_seed(seed)

seed_everything(42)

# ── Load dataset ──────────────────────────────────────────────────────────────
file_path = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'

# Fallback for local run
if not os.path.exists(file_path):
    file_path = 'train.csv'

dataset = load_dataset('csv', data_files=file_path)['train']
train_df = pd.read_csv(file_path)
print("Dataset loaded successfully!")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset loaded successfully!


---
## Q1. Label Encoding
Convert the answer column into numeric labels (A=0, B=1, C=2, D=3, E=4).
What is the encoded numeric label for the row at index 150?

In [4]:
# ── Q1 ────────────────────────────────────────────────────────────────────────
label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
q1_ans = label_map[dataset[150]['answer']]
print(f">>> Q1 Answer: {q1_ans}")

>>> Q1 Answer: 2


---
## Q2. Prompt-Option Formatting
For row index 0, create the Option B input using format: `str(prompt) + " [SEP] " + str(option_B)`.
What is the exact character length?

In [5]:
# ── Q2 ────────────────────────────────────────────────────────────────────────
prompt_0 = str(dataset[0]['prompt'])
option_B_0 = str(dataset[0]['B'])
formatted_q2 = prompt_0 + " [SEP] " + option_B_0
q2_ans = len(formatted_q2)
print(f">>> Q2 Answer: {q2_ans}")

>>> Q2 Answer: 407


---
## Q3-Q6 Conceptual Questions

In [6]:
# ── Q3-Q6 ──────────────────────────────────────────────────────────────────────
print(">>> Q3 Answer (Second dimension for num_choices): 5")
print(">>> Q4 Answer (Total token positions 16*5*128): 10240")
print(">>> Q5 Answer (Number of logits produced): 5")
print(">>> Q6 Answer (Dimensions of scalar loss tensor): 0")

>>> Q3 Answer (Second dimension for num_choices): 5
>>> Q4 Answer (Total token positions 16*5*128): 10240
>>> Q5 Answer (Number of logits produced): 5
>>> Q6 Answer (Dimensions of scalar loss tensor): 0


---
## Q7. LoRA Trainable Parameters
Apply LoRA using r=8, alpha=16, modules=["query", "value"]. How many trainable parameters?

In [10]:
# ── Q7 ────────────────────────────────────────────────────────────────────────
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r=8, 
    lora_alpha=16, 
    target_modules=["query", "value"], 
    lora_dropout=0.1, 
    bias="none"
    # task_type=TaskType.SEQ_CLS is removed to avoid the torchao version check mismatch
)

peft_model = get_peft_model(model, lora_config)
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f">>> Q7 Answer (Trainable parameters): {trainable_params}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Skipping import of cpp extensions due to inco

>>> Q7 Answer (Trainable parameters): 294912


---
## Q8. Hugging Face Dataset Preparation
How many tokenized choices are stored in input_ids for the first dataset item?

In [11]:
# ── Q8 ────────────────────────────────────────────────────────────────────────
print(">>> Q8 Answer (Tokenized choices in input_ids): 5")

>>> Q8 Answer (Tokenized choices in input_ids): 5


---
## Q9 & Q10. Tiny Fine-Tuning and Inference
Fine-tune on the first 32 rows, max_steps=4. Then run inference on row index 0 to get probability of Option E.

In [13]:
# ── Q9 & Q10 ──────────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def preprocess_function(examples):
    first_sentences = [[context] * 5 for context in examples["prompt"]]
    second_sentences = [
        [f"{examples[option][i]}" for option in ["A", "B", "C", "D", "E"]] for i in range(len(examples["prompt"]))
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized_examples = tokenizer(
        first_sentences,
        second_sentences,
        truncation=True,
        max_length=64,
        padding="max_length"
    )
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized_examples.items()}

# Prepare dataset
subset_dataset = dataset.select(range(32))
subset_dataset = subset_dataset.map(lambda x: {'labels': label_map[x['answer']]})
tokenized_dataset = subset_dataset.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns([c for c in dataset.column_names if c != 'labels'])
tokenized_dataset.set_format("torch")

# Force CPU/GPU correctly
device = "cuda" if torch.cuda.is_available() else "cpu"
peft_model.to(device)

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
    learning_rate=5e-5,
    logging_steps=1,
    report_to="none"
)

data_collator = DefaultDataCollator()

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,  # Changed from tokenizer=tokenizer
    data_collator=data_collator
)

trainer.train()
q9_ans = trainer.state.global_step
print(f"\n>>> Q9 Answer (Final global_step): {q9_ans}")

# Inference
with torch.no_grad():
    peft_model.eval()
    input_ids = tokenized_dataset[0]["input_ids"].unsqueeze(0).to(device)
    attention_mask = tokenized_dataset[0]["attention_mask"].unsqueeze(0).to(device)
    token_type_ids = tokenized_dataset[0]["token_type_ids"].unsqueeze(0).to(device)
    
    logits = peft_model(
        input_ids=input_ids, 
        attention_mask=attention_mask,
        token_type_ids=token_type_ids
    ).logits
    probs = torch.softmax(logits, dim=-1)
    q10_ans = round(probs[0, 4].item(), 4)

print(f">>> Q10 Answer (Probability of Option E): {q10_ans}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss
1,1.658626
2,1.569909
3,1.598525
4,1.653382



>>> Q9 Answer (Final global_step): 4
>>> Q10 Answer (Probability of Option E): 0.1996
